# 01 · LLM Providers: Choosing the "Brains" of Our Agents

**Goal of this notebook:** pick the language model(s) every agent in the research system will use.

A multi-agent system makes *many* LLM calls per research query (planning, one ReAct loop per researcher,
fact-checking, writing, critiquing). So the model choice affects:

| Concern | Why it matters for us |
|---|---|
| **Latency** | Calls run one after another (planner → researchers → writer → critic). Slow models multiply. |
| **Tool calling** | Researchers must *decide* to call search tools and emit valid arguments. |
| **Structured output** | Planner, critic, etc. must return data that matches a Pydantic schema, not free text. |
| **Rate limits / cost** | Free tiers cap requests per minute/day; one research run can use 20–40 calls. |
| **Reliability** | Providers fail. We need a fallback path. |

### Providers we use
- **OpenRouter**: one API key, hundreds of models (OpenAI, Google, Meta, NVIDIA, ...). Models ending in `:free`
  cost nothing but are rate limited (≈20 req/min, ≈50 req/day on accounts without credits).
- **Groq** (optional, plug-in later): runs open models on custom LPU chips, so it is extremely fast.
  The code below automatically includes Groq models once `GROQ_API_KEY` is set.

### What we will test
1. A basic call and what comes back (`AIMessage`, token usage, metadata).
2. A small benchmark across candidate models: latency, tool calling, structured output.
3. Temperature: deterministic vs creative output.
4. Streaming tokens.
5. **Fallbacks**: automatically switching to a backup model when the primary fails.
6. A provider-agnostic **model factory**, the pattern we will reuse in production code.

## 0 · Setup

`nb_setup.setup()` loads keys from `.env`, shows which are present, and enables the **LLM cache**
(`data/llm_cache.sqlite`). With the cache on, re-running a cell with the same prompt and model costs **zero API
requests**, which is essential on a free tier. We switch the cache *off* only when measuring real latency.

In [1]:
from nb_setup import setup, disable_cache, timer

ROOT = setup()

API keys:
  GROQ_API_KEY         MISSING
  OPENROUTER_API_KEY   set
  TAVILY_API_KEY       set
  LANGSMITH_API_KEY    set
  LangSmith tracing    true


LLM cache: data\llm_cache.sqlite


## 1 · The model factory (provider-agnostic design)

Agents should never care *which* provider serves them. They receive a LangChain **chat model** object, and every
provider's class (`ChatOpenRouter`, `ChatGroq`, ...) implements the same interface (`BaseChatModel`):
`.invoke()`, `.stream()`, `.bind_tools()`, `.with_structured_output()`, `.with_fallbacks()`.

So we hide provider details behind one small function. Swapping providers then becomes a config change, not a
code change. This is the **Factory pattern**, and the same function moves into `src/research_system/config/llm.py`
in Phase 2.

Key parameters:
- `temperature=0`: make outputs as deterministic as possible (good for planning/structured tasks).
- `max_retries`: the client retries transient errors (HTTP 429/5xx) with backoff.
- `timeout`: never let a hung request block the whole graph.

> ⚠️ **A real gotcha we hit while building this:** `ChatOpenRouter(timeout=...)` is in **milliseconds**, while
> `ChatGroq` (and most LangChain clients) use **seconds**. Passing `timeout=20` to OpenRouter means *20 ms*: every
> request times out instantly and the SDK keeps retrying in the background, so the call appears to "hang" for
> minutes. A factory is the right place to hide this kind of inconsistency: callers always pass **seconds**.

In [2]:
import os
from langchain_core.language_models.chat_models import BaseChatModel


def make_model(provider: str, model: str, temperature: float = 0.0,
               max_retries: int = 1, timeout: float = 60) -> BaseChatModel:
    """Create a chat model for the given provider. `timeout` is always in seconds."""
    if provider == "openrouter":
        from langchain_openrouter import ChatOpenRouter
        return ChatOpenRouter(model=model, temperature=temperature,
                              max_retries=max_retries, timeout=int(timeout * 1000))  # SDK wants ms
    if provider == "groq":
        from langchain_groq import ChatGroq
        return ChatGroq(model=model, temperature=temperature,
                        max_retries=max_retries, timeout=timeout)
    raise ValueError(f"Unknown provider: {provider}")


# Candidate models. Free OpenRouter models change over time; the live list is at
# https://openrouter.ai/models?max_price=0  (filter: supports tools).
CANDIDATES = [
    ("openrouter", "nvidia/nemotron-3.5-super-vl:free"),
    ("openrouter", "inclusionai/ling-3.2-flash:free"),
    ("openrouter", "baidu/ernie-5.5-flash-preview:free"),
    ("openrouter", "google/gemma-4-31b-it:free"),   # popular -> often rate-limited upstream
]

# Groq candidates are added only when a key exists (verify ids at console.groq.com/docs/models).
if os.getenv("GROQ_API_KEY"):
    CANDIDATES += [("groq", "llama-3.3-70b-versatile"), ("groq", "llama-3.1-8b-instant")]

CANDIDATES

[('openrouter', 'nvidia/nemotron-3.5-super-vl:free'),
 ('openrouter', 'inclusionai/ling-3.2-flash:free'),
 ('openrouter', 'baidu/ernie-5.5-flash-preview:free'),
 ('openrouter', 'google/gemma-4-31b-it:free')]

## 2 · Anatomy of a single call

`model.invoke(...)` accepts a string or a list of **messages**:
- `SystemMessage`: instructions/persona ("You are a research planner...").
- `HumanMessage`: the user's input.
- `AIMessage`: the model's reply (also used to replay conversation history).

The reply is an `AIMessage` with:
- `.content`: the text.
- `.usage_metadata`: input/output token counts (used later for cost tracking).
- `.response_metadata`: provider details (model actually used, finish reason, ...).
- `.tool_calls`: filled when the model decides to call a tool (section 4).

In [3]:
from langchain_core.messages import HumanMessage, SystemMessage

provider, model_id = CANDIDATES[0]
llm = make_model(provider, model_id)

messages = [
    SystemMessage("You are a concise research assistant. Answer in at most two sentences."),
    HumanMessage("What problem does LangGraph solve compared with a plain LangChain chain?"),
]

with timer(model_id):
    reply = llm.invoke(messages)

print(type(reply).__name__)
print("content :", reply.content)
print("usage   :", reply.usage_metadata)
print("finish  :", reply.response_metadata.get("finish_reason"))

[nvidia/nemotron-3.5-super-vl:free] 0.12s
AIMessage
content : LangGraph enables stateful, cyclic workflows with loops and conditional branching, whereas plain LangChain chains are typically linear or acyclic and lack persistent state across steps. This makes LangGraph better suited for complex agentic systems requiring iteration, memory, and human-in-the-loop control.
usage   : {'input_tokens': 44, 'output_tokens': 272, 'total_tokens': 316, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}, 'output_token_details': {'reasoning': 214}, 'total_cost': 0}
finish  : stop


## 3 · Benchmark harness

We test every candidate on three capabilities the agents depend on. Each test returns a small dict so the results
fit in one comparison table.

**Test A: plain answer (latency + quality).** A short factual research question.

**Test B: tool calling.** We give the model a tool *schema* (name, description, typed arguments) and a task that
needs it. A capable model replies with a `tool_call` (it does *not* run the tool itself; our code does that later).
We check that it chose the right tool and produced valid arguments.

**Test C: structured output.** We ask for a `ResearchPlan`-like Pydantic object. `with_structured_output()` makes
the model return JSON matching the schema, and LangChain validates and parses it into the Pydantic class.
This is how the Planner and Critic agents will talk to the graph.

In [4]:
from pydantic import BaseModel, Field
from langchain_core.tools import tool


@tool
def search_papers(query: str, max_results: int = 5) -> str:
    """Search academic papers (arXiv) for a query and return titles with abstracts."""
    return "stub"  # never executed here, we only inspect the model's tool *request*


class MiniPlan(BaseModel):
    """A research plan: the main objective broken into focused sub-questions."""
    objective: str = Field(description="One-sentence research objective")
    sub_questions: list[str] = Field(description="3 to 5 focused, searchable sub-questions")


QUESTION = "What are the main techniques for reducing hallucinations in large language models?"


def test_plain(llm) -> dict:
    with timer("plain") as t:
        r = llm.invoke(f"Answer in 3 bullet points: {QUESTION}")
    return {"plain_s": round(t["seconds"], 1), "plain_tokens": (r.usage_metadata or {}).get("total_tokens"),
            "plain_preview": r.content[:80].replace("\n", " ")}


def test_tools(llm) -> dict:
    r = llm.bind_tools([search_papers]).invoke("Find recent arXiv papers about retrieval-augmented generation.")
    calls = r.tool_calls
    ok = bool(calls) and calls[0]["name"] == "search_papers" and "query" in calls[0]["args"]
    return {"tool_call_ok": ok, "tool_args": calls[0]["args"] if calls else None}


def test_structured(llm) -> dict:
    try:
        plan = llm.with_structured_output(MiniPlan).invoke(f"Create a research plan for: {QUESTION}")
        ok = isinstance(plan, MiniPlan) and 3 <= len(plan.sub_questions) <= 5
        return {"structured_ok": ok, "n_subq": len(plan.sub_questions)}
    except Exception as e:  # malformed JSON / validation error
        return {"structured_ok": False, "n_subq": None, "structured_err": type(e).__name__}

### Run the benchmark

Latency only means something without the cache, so we turn it off for this cell. Each candidate costs ~3 requests.
Every test is wrapped in `try/except`: free models are sometimes overloaded (HTTP 429/503), and **one failing model
must not stop the experiment**. That is the same principle our graph will follow.

Because a full benchmark spends ~12 of the ~50 free daily requests, results are saved to
`data/01_model_benchmark.csv` and **re-used on later runs**. Set `RERUN_BENCHMARK = True` to measure again
(for example after adding a Groq key).

In [5]:
import pandas as pd

RERUN_BENCHMARK = False
BENCH_CSV = ROOT / "data" / "01_model_benchmark.csv"

if BENCH_CSV.exists() and not RERUN_BENCHMARK:
    results = pd.read_csv(BENCH_CSV)
    print("Loaded saved benchmark (set RERUN_BENCHMARK = True to re-measure)")
else:
    disable_cache()
    rows = []
    for provider, model_id in CANDIDATES:
        bench_llm = make_model(provider, model_id, max_retries=1)
        row = {"provider": provider, "model": model_id}
        for test in (test_plain, test_tools, test_structured):
            try:
                row |= test(bench_llm)
            except Exception as e:
                row[f"{test.__name__}_error"] = f"{type(e).__name__}: {str(e)[:60]}"
        rows.append(row)
    results = pd.DataFrame(rows)
    results.to_csv(BENCH_CSV, index=False)
    setup()  # re-enable the cache for the rest of the notebook
    print("saved ->", BENCH_CSV.relative_to(ROOT))

results[["model", "plain_s", "plain_tokens", "tool_call_ok", "structured_ok", "n_subq"]]

Loaded saved benchmark (set RERUN_BENCHMARK = True to re-measure)


,model,plain_s,plain_tokens,tool_call_ok,structured_ok,n_subq
0,nvidia/nemotron-3.5-super-vl:free,4.7,381.0,True,True,5.0
1,inclusionai/ling-3.2-flash:free,38.2,543.0,True,True,5.0
2,baidu/ernie-5.5-flash-preview:free,1.8,210.0,True,True,5.0
3,google/gemma-4-31b-it:free,NaN,NaN,NaN,False,NaN


## 4 · Temperature: determinism vs creativity

`temperature` scales the randomness of token sampling.
- **0.0**: (nearly) always picks the most likely token. Repeatable, good for planning, extraction, JSON, grading.
- **~0.7–1.0**: more varied wording. Useful for brainstorming search queries or writing prose.

Plan for the system: planner / fact-checker / critic at **0**, writer at **~0.3–0.5** for natural prose.

In [6]:
provider, model_id = CANDIDATES[0]
prompt = "Suggest one creative title for a report on AI agents. Reply with the title only."
for temp in (0.0, 1.0):
    print(f"T={temp}:", make_model(provider, model_id, temperature=temp).invoke(prompt).content.strip())

T=0.0: The Agentic Era: AI Agents That Think, Act, and Deliver
T=1.0: Agentic Intelligence: The Rise of Autonomous AI Agents


## 5 · Streaming

`.stream()` yields `AIMessageChunk`s as tokens are generated. The UI (Phase 4) will stream the Writer's report this way so
users see progress instead of waiting 30+ seconds for a blank screen.

In [7]:
# Streaming bypasses the cache, so this cell always makes one real request.
stream_llm = make_model("openrouter", "baidu/ernie-5.5-flash-preview:free")
for chunk in stream_llm.stream("In two sentences, explain what a multi-agent system is."):
    print(chunk.content, end="", flush=True)

A multi-agent system is a computational framework in which multiple autonomous agents, each with its own goals and capabilities, interact within a shared environment. These agents may cooperate, compete, or coordinate

 to solve problems that are difficult for a single agent, often producing emergent or distributed behavior.

## 6 · Fallbacks: surviving provider failures

`primary.with_fallbacks([backup, ...])` returns a new runnable: if the primary raises an exception (rate limit,
timeout, outage), LangChain transparently retries the **same input** on the next model.

To prove it works we create a primary that is *guaranteed* to fail (a model id that doesn't exist) and watch the
backup answer. In production the chain will be: Groq model → OpenRouter model A → OpenRouter model B.

In [8]:
broken = make_model("openrouter", "this-model/does-not-exist", max_retries=0)
backup = make_model(*CANDIDATES[0])

resilient = broken.with_fallbacks([backup])
reply = resilient.invoke("Reply with exactly: fallback works")
print("answered by fallback ->", reply.content, "|", reply.response_metadata.get("model_name"))

answered by fallback -> fallback works | nvidia/nemotron-3.5-super-vl:free


## 7 · Observations & Decisions

*Recorded from the run on 2026-11-18 (OpenRouter free tier, no Groq key yet). Raw table: `data/01_model_benchmark.csv`.*

| Model | Plain answer | Tool call | Structured output | Notes |
|---|---|---|---|---|
| `nvidia/nemotron-3.5-super-vl:free` | 4.7 s | ✅ | ✅ (5 sub-questions) | Reasoning model: ~80% of output tokens were hidden "reasoning" tokens |
| `baidu/ernie-5.5-flash-preview:free` | **1.8 s** | ✅ | ✅ | Fastest and most token-efficient (210 tokens) |
| `inclusionai/ling-3.2-flash:free` | 38.2 s | ✅ | ✅ | Correct but very slow (upstream queueing) |
| `google/gemma-4-31b-it:free` | n/a | n/a | n/a | HTTP 429: rate-limited upstream on every attempt |

**Observations**
- All models that responded passed **both** tool calling and structured output. On today's models, capability is
  not the bottleneck; **latency and availability** are.
- Free-model availability is volatile: a popular model (Gemma) was unusable while less popular ones answered quickly.
- Reasoning models (Nemotron) spend many tokens "thinking". Better for planning and critique, wasteful for simple extraction.
- `ChatOpenRouter` takes its timeout in **milliseconds**. The factory converts it, and the production config must too.
- Fallbacks work: a broken primary model was transparently replaced by the backup.

**Decisions**
1. Use the **`make_model()` factory**; agents receive a `BaseChatModel`, never a provider class.
2. **Model roles** (all set in config; revisit when a Groq key is added):
   - *Reasoning roles* (planner, fact-checker, critic): `nvidia/nemotron-3.5-super-vl:free`, T=0.
   - *Worker roles* (researcher ReAct loop, summarising, writer): `baidu/ernie-5.5-flash-preview:free` (writer at T≈0.3).
   - *Fallback for both*: the other one, then `inclusionai/ling-3.2-flash:free` (slow but reliable).
3. **Always** wrap models with `.with_fallbacks()`; low client retries (`max_retries=1`) so a rate-limited model fails
   fast and the fallback takes over, rather than waiting through long backoffs.
4. Keep the **LLM cache** on in notebooks; record benchmarks to CSV instead of re-running them.